In [34]:
try:
    !pip install -q \
    chromadb \
    google-genai \
    groq

    print("Libraries installed successfully!")

except Exception as e:
    print("Installation Error:")
    print(e)

Libraries installed successfully!


In [35]:
try:

    import pickle

    with open("chunks.pkl", "rb") as f:
        chunks = pickle.load(f)

    with open("chunk_embeddings.pkl", "rb") as f:
        chunk_embeddings = pickle.load(f)

    print("Files loaded successfully!")

except Exception as e:

    print("Loading Error:")
    print(e)

Files loaded successfully!


In [36]:
try:

    import chromadb

    chroma_client = chromadb.PersistentClient(
        path="./networking_chromadb"
    )

    collection = chroma_client.get_or_create_collection(
        name="networking_docs"
    )

    collection.add(
        ids=[f"chunk_{i}" for i in range(len(chunks))],
        documents=chunks,
        embeddings=chunk_embeddings
    )

    print("Collection recreated successfully!")
    print("Records:", collection.count())

except Exception as e:

    print(e)

Collection recreated successfully!
Records: 182


In [37]:
try:

    from google import genai
    from google.colab import userdata

    client = genai.Client(
        api_key=userdata.get("GEMINI_API_KEY")
    )

    print("Gemini initialized successfully!")

except Exception as e:

    print("Gemini Error:")
    print(e)

Gemini initialized successfully!


In [38]:
try:

    query = "What is TCP?"

    response = client.models.embed_content(
        model="models/gemini-embedding-001",
        contents=query
    )

    query_embedding = response.embeddings[0].values

    print("Query embedding generated!")

except Exception as e:

    print("Embedding Error:")
    print(e)

Query embedding generated!


In [39]:
try:

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=5
    )

    print("Retrieved top 5 chunks!")

except Exception as e:

    print("Retrieval Error:")
    print(e)

Retrieved top 5 chunks!


In [40]:
try:

    retrieved_chunks = results["documents"][0]

    for i, chunk in enumerate(retrieved_chunks):

        print("=" * 60)
        print(f"Chunk {i + 1}")
        print(chunk[:500])
        print()

except Exception as e:

    print("Display Error:")
    print(e)

Chunk 1
The Transmission Control Protocol (TCP) is one of the main protocols of the Internet protocol suite, providing reliable, ordered, and error-checked delivery of a stream of octets (bytes) between applications running on hosts communicating via an IP network. It originated in the initial network implementation in which it complemented the Internet Protocol (IP). Therefore, the entire suite is commonly referred to as TCP/IP.
Major internet applications such as the World Wide Web, email, remote admi

Chunk 2
Network function
The Transmission Control Protocol provides a communication service at an intermediate level between an application program and the Internet Protocol. It provides host-to-host connectivity at the transport layer of the Internet model. An application does not need to know the particular mechanisms for sending data via a link to another host, such as the required IP fragmentation to accommodate the maximum transmission unit of the transmission medium. At the trans

In [41]:
try:

    from groq import Groq
    from google.colab import userdata

    groq_client = Groq(
        api_key=userdata.get("GROQ_API_KEY")
    )

    print("Groq initialized successfully!")

except Exception as e:

    print("Groq Error:")
    print(e)

Groq initialized successfully!


In [42]:
try:

    context = "\n\n".join(retrieved_chunks)

    print("Context created successfully!")

except Exception as e:

    print("Context Error:")
    print(e)

Context created successfully!


In [43]:
try:

    prompt = f"""
You are a networking assistant.

Answer only using the provided context.

If the answer is not found, reply:

I could not find that information in the knowledge base.

Context:
{context}

Question:
{query}
"""

    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    answer = response.choices[0].message.content

    print(answer)

except Exception as e:

    print("Generation Error:")
    print(e)

The Transmission Control Protocol (TCP) is one of the main protocols of the Internet protocol suite, providing reliable, ordered, and error-checked delivery of a stream of octets (bytes) between applications running on hosts communicating via an IP network.


In [44]:
def ask_network_question(question):

    try:

        embedding_response = client.models.embed_content(
            model="models/gemini-embedding-001",
            contents=question
        )

        query_embedding = embedding_response.embeddings[0].values

        search_results = collection.query(
            query_embeddings=[query_embedding],
            n_results=5
        )

        context = "\n\n".join(
            search_results["documents"][0]
        )

        prompt = f"""
You are a networking assistant.

Answer only using the provided context.

Context:
{context}

Question:
{question}
"""

        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )

        return response.choices[0].message.content

    except Exception as e:

        return f"Error: {e}"

In [45]:
print(
    ask_network_question(
        "What is the purpose of DNS?"
    )
)

The purpose of DNS is to translate human-friendly computer hostnames into IP addresses, serving as a phone book for the Internet. It also provides other functions such as mapping domain names to Internet resources, facilitating mail delivery, and supporting distributed Internet services like cloud services and content delivery networks. Additionally, DNS helps in efficient storage and distribution of IP addresses of block-listed email hosts, and provides resilience in the event of computer or network failure.


In [46]:
print(
    ask_network_question(
        "What is the difference between TCP and UDP?"
    )
)

The main differences between TCP and UDP are:

1. **Reliability**: TCP is reliable, ensuring that data arrives in-order and with minimal error, whereas UDP is unreliable, with no guarantee of delivery.
2. **Connection**: TCP is connection-oriented, requiring a handshake to set up a connection, whereas UDP is connectionless, with no prior setup needed.
3. **Order**: TCP ensures that data arrives in-order, whereas UDP does not guarantee the order of arrival.
4. **Error control**: TCP has built-in error control, whereas UDP relies on error detection using a checksum algorithm.
5. **Congestion control**: TCP has congestion control, whereas UDP does not and relies on the application or network to handle congestion.
6. **Overhead**: TCP has more overhead due to the handshake and error control, whereas UDP is lightweight with less overhead.

In general, TCP is used for applications that require reliable data transfer, such as file transfers and email, whereas UDP is used for applications that